# Do LLMs Verify or Conform? — 实验原理与内容解析

这个 notebook 逐步解释我们研究的每一个环节：**为什么做、在测什么、怎么测、结果意味着什么**。

所有例子都来自真实数据（aws-c-common，83 个函数，gpt-oss-120b）。

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

BASE = '/home/weiqi/Verification/LLM4Harness/experiment_aws_cbmc'
RESULTS = os.path.join(BASE, 'results')
EVAL = os.path.join(BASE, 'evaluation')

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 11
print('Setup complete.')

---
## Part 1：背景 — CBMC 是什么

CBMC（C Bounded Model Checker）是一个**静态程序分析工具**。给它一段 C 代码和一些断言（`assert`），它会：

- **UNSAT（SUCCESS）**：在有界的状态空间里，**所有可能的输入**都不会违反 `assert`。你的断言对所有情况都成立。
- **SAT（FAIL）**：找到了一个具体的反例——某个输入让某个 `assert` 失败了。
- **UNKNOWN / TIMEOUT**：状态空间太大了，CBMC 探索不完。没有结论。

> 与普通测试的区别：普通测试跑有限个输入。CBMC 证明或证伪**所有有界输入**。

In [ ]:
# 可视化 CBMC 的三种结果
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

scenarios = [
    {
        'title': 'CBMC → SUCCESS\n(UNSAT)',
        'color': '#2ecc71',
        'icon': '✓',
        'desc': '所有输入路径都被探索\n没有任何 assert 被违反\n→ 断言在有界范围内正确',
        'detail': 'assert(result >= 0) ✓\nassert(list.length == old.length + 1) ✓\nassert(ptr != NULL) ✓'
    },
    {
        'title': 'CBMC → FAIL\n(SAT)',
        'color': '#e74c3c',
        'icon': '✗',
        'desc': '找到了一个反例\n某个具体输入让 assert 失败\n→ 代码有 bug 或断言写错了',
        'detail': 'Counterexample:\n  list.length = 5, index = 7\n  assert(index < list.length)  ← FAILS'
    },
    {
        'title': 'CBMC → UNKNOWN\n(Timeout)',
        'color': '#f39c12',
        'icon': '?',
        'desc': '状态空间爆炸\n在时间限制内无法完成\n→ 没有结论',
        'detail': 'State space too large.\nUnbounded loops or\ntoo many pointer dereferences.'
    }
]

for ax, s in zip(axes, scenarios):
    ax.set_facecolor(s['color'] + '22')
    ax.add_patch(mpatches.FancyBboxPatch((0.05, 0.05), 0.9, 0.9,
        boxstyle='round,pad=0.02', linewidth=2,
        edgecolor=s['color'], facecolor=s['color'] + '15'))
    ax.text(0.5, 0.85, s['title'], ha='center', va='top', fontsize=13,
            fontweight='bold', color=s['color'], transform=ax.transAxes)
    ax.text(0.5, 0.60, s['desc'], ha='center', va='top', fontsize=9.5,
            transform=ax.transAxes, linespacing=1.6)
    ax.text(0.5, 0.30, s['detail'], ha='center', va='top', fontsize=8.5,
            transform=ax.transAxes, family='monospace',
            color='#333', linespacing=1.5)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

fig.suptitle('CBMC 的三种可能输出', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Part 2：Proof Harness 是什么

CBMC 不能直接对整个程序做验证。你需要为每个函数写一个**proof harness**（证明框架）。

Harness 做三件事：
1. **建立有效输入**：用 `__CPROVER_assume()` 约束输入在合法范围内
2. **调用被验证函数**
3. **写出你想验证的属性**：用 `assert()` 表达"函数调用后，这些事情必须成立"

下面是 AWS 工程师为 `aws_array_list_erase` 写的 GT（Ground Truth）harness：

In [ ]:
gt_harness = '''
void aws_array_list_erase_harness() {
    struct aws_array_list list;   // 声明一个 array list
    size_t index;                 // 要删除的位置（非确定性）

    // ① ASSUMES：告诉 CBMC 只考虑有效输入
    __CPROVER_assume(aws_array_list_is_bounded(&list, MAX_INITIAL_ITEM_ALLOCATION, MAX_ITEM_SIZE));
    ensure_array_list_has_allocated_data_member(&list);
    __CPROVER_assume(aws_array_list_is_valid(&list));

    // ② 保存调用前状态（用来和调用后对比）
    struct aws_array_list old = list;
    struct store_byte_from_buffer old_byte;
    save_byte_from_array((uint8_t *)list.data, list.current_size, &old_byte);

    // ③ 调用被验证函数
    if (aws_array_list_erase(&list, index) == AWS_OP_SUCCESS) {

        // ④ ASSERTS（成功时的后置条件）
        assert(list.length == old.length - 1);        // 长度减一
        assert(list.item_size == old.item_size);       // item_size 不变
        assert(list.alloc == old.alloc);               // 分配器不变
        assert(list.current_size == old.current_size); // 容量不变
        assert(index < old.length);                    // 索引必须合法

    } else {
        // ④' 失败时：列表应该没有变化
        assert_array_list_equivalence(&list, &old, &old_byte);
    }
    // 无论成败，数据结构必须有效
    assert(aws_array_list_is_valid(&list));
}
'''

print('AWS 工程师写的 GT Harness (aws_array_list_erase):')
print(gt_harness)

print('\n共有断言 (assert):',
      gt_harness.count('assert(') + gt_harness.count('assert_array'), '条')
print('共有假设 (assume):', gt_harness.count('__CPROVER_assume'), '条')

---
## Part 3：实验流程 — LLM 怎么生成 Harness

我们让 LLM（gpt-oss-120b）生成 harness，然后用 CBMC 检验，根据结果给 LLM 反馈，循环最多 15 次：

```
LLM 生成 iter_1_harness.c
        ↓
   CBMC 运行
     ↙    ↘      ↘
 SUCCESS   FAIL   UNKNOWN
   ↓        ↓        ↓
  完成   "修复这个断言"  "状态空间太大，简化一下"
              ↓        ↓
          LLM 生成 iter_2_harness.c
              ↓
           继续...
```

关键问题：**CBMC 返回 UNKNOWN 时，LLM 会怎么做？**

In [ ]:
# 可视化迭代流程
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('off')

# 绘制流程框
boxes = [
    (0.08, 0.5, 'LLM\n生成 iter_1', '#3498db', 'white'),
    (0.28, 0.5, 'CBMC\n运行', '#95a5a6', 'white'),
    (0.48, 0.82, 'SUCCESS\n✓', '#2ecc71', 'white'),
    (0.48, 0.5,  'FAIL\n(反例)', '#e74c3c', 'white'),
    (0.48, 0.18, 'UNKNOWN\n(超时)', '#f39c12', 'white'),
    (0.72, 0.5, 'LLM\n生成 iter_N+1', '#3498db', 'white'),
    (0.92, 0.5, '完成\n(PASS)', '#2ecc71', 'white'),
]

for x, y, label, color, tc in boxes:
    ax.add_patch(mpatches.FancyBboxPatch((x-0.07, y-0.12), 0.14, 0.24,
        boxstyle='round,pad=0.01', linewidth=2,
        edgecolor=color, facecolor=color + '33'))
    ax.text(x, y, label, ha='center', va='center', fontsize=10,
            fontweight='bold', color=color)

# 箭头
arrows = [
    (0.15, 0.5, 0.21, 0.5, ''),
    (0.35, 0.56, 0.41, 0.76, 'UNSAT'),
    (0.35, 0.5,  0.41, 0.5,  'SAT→修复断言'),
    (0.35, 0.44, 0.41, 0.24, 'UNKNOWN→简化'),
    (0.55, 0.5,  0.65, 0.5,  ''),
    (0.55, 0.24, 0.65, 0.44, ''),
    (0.79, 0.5,  0.85, 0.5,  ''),
]
for x1, y1, x2, y2, label in arrows:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
    if label:
        ax.text((x1+x2)/2, (y1+y2)/2 + 0.04, label,
                ha='center', fontsize=8.5, color='#555',
                style='italic')

# "UNKNOWN" 时的关键问题
ax.text(0.5, 0.02,
    '⚠️  关键问题：面对 UNKNOWN，LLM 有两种策略：'
    '(A) 修复 assume 让状态空间变小  or  (B) 删掉 assert 让 CBMC 有东西可以验证',
    ha='center', fontsize=9.5, color='#c0392b',
    fontweight='bold')

ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('LLM-CBMC 迭代循环', fontsize=14, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

---
## Part 4：真实的 Sacrifice 案例 — aws_array_list_erase

这是我们观察到的现象：LLM 在 CBMC UNKNOWN 后，**删掉了 8 个断言**。

In [ ]:
func = 'aws_array_list_erase'
result_dir = os.path.join(RESULTS, 'feedback_loop_A_gptoss120b', func)

with open(os.path.join(result_dir, 'summary.json')) as f:
    summary = json.load(f)

# 读取每次迭代的 harness
iter_files = sorted([f for f in os.listdir(result_dir)
                     if f.startswith('iter_') and f.endswith('_harness.c')])

iter_asserts = {}
iter_code = {}
for fname in iter_files:
    num = int(fname.split('_')[1])
    with open(os.path.join(result_dir, fname)) as f:
        code = f.read()
    iter_code[num] = code
    # 数 assert 行（排除 __CPROVER_assume 和 #include）
    n = sum(1 for line in code.splitlines()
            if 'assert(' in line and '__CPROVER_assume' not in line
            and not line.strip().startswith('//'))
    iter_asserts[num] = n

print(f'函数: {func}')
print(f'迭代次数: {len(summary["iterations"])}')
print()
for it in summary['iterations']:
    n = iter_asserts.get(it['iter'], '?')
    status_color = {'SUCCESS': '✅', 'FAIL': '❌', 
                    'UNKNOWN': '⚠️', 'COMPILE_ERROR': '🔧'}.get(it['verify'], '?')
    print(f"  Iter {it['iter']}: {status_color} {it['verify']:15s}  "
          f"assert 数量: {n}  action: {it['action']}")

In [ ]:
# 对比 iter_1 和 iter_2 的断言
def extract_asserts(code):
    lines = []
    for line in code.splitlines():
        stripped = line.strip()
        if ('assert(' in stripped and
            '__CPROVER_assume' not in stripped and
            not stripped.startswith('//') and
            '#include' not in stripped):
            lines.append(stripped)
    return lines

asserts_1 = extract_asserts(iter_code[1])
asserts_2 = extract_asserts(iter_code[2])
set_1 = set(asserts_1)
set_2 = set(asserts_2)

print('='*65)
print(f'ITER 1 断言（{len(asserts_1)} 条）→ CBMC: UNKNOWN')
print('='*65)
for a in asserts_1:
    marker = '❌ DELETED' if a not in set_2 else '  kept   '
    print(f'  {marker}  {a}')

print()
print('='*65)
print(f'ITER 2 断言（{len(asserts_2)} 条）→ CBMC: SUCCESS')
print('='*65)
for a in asserts_2:
    marker = '✨ NEW' if a not in set_1 else '       '
    print(f'  {marker}  {a}')

print()
print(f'删除了 {len(set_1 - set_2)} 条断言，新增了 {len(set_2 - set_1)} 条断言')

In [ ]:
# 与 GT 对比：哪些 GT 断言在 iter_1 存在但被删了，哪些从未生成过
gt_asserts_raw = [
    'assert(list.length == old.length - 1)',
    'assert(list.item_size == old.item_size)',
    'assert(list.alloc == old.alloc)',
    'assert(list.current_size == old.current_size)',
    'assert(index < old.length)',
    'assert_array_list_equivalence(&list, &old, &old_byte)',
    'assert(aws_array_list_is_valid(&list))',
]

print('GT Harness 断言 vs LLM 最终 Harness (iter_2):')
print()
final_code = '\n'.join(asserts_2)
for gt in gt_asserts_raw:
    # 宽松匹配：检查关键词是否出现
    key = gt.replace('assert(', '').replace(')', '').strip().split('==')[0].strip()
    appeared_any = any(key in a for a in asserts_1 + asserts_2)
    in_final = any(key in a for a in asserts_2)
    if in_final:
        status = '✅ 匹配'
    elif appeared_any:
        status = '⚠️ 曾生成后删除 (SACRIFICE)'
    else:
        status = '❌ 从未生成 (KNOWLEDGE GAP)'
    print(f'  {status}  {gt}')

---
## Part 5：PASS 率 vs Recall — 为什么 PASS 率是误导性指标

**PASS 率**：LLM harness 通过 CBMC 验证的函数比例。  
**Recall**：LLM harness 中，与 GT 断言**匹配的比例**（在 GT assumes 下重新验证）。

核心发现：**PASS 率越高的条件，Recall 不一定越高，甚至可能更低！**

In [ ]:
# 实验结果数据（来自 research_design.md，所有数值已在统计检验中验证）
conditions_data = {
    'G\n(无反馈)':      {'pass': 31.3, 'recall': 0.290, 'sacrifice': 0.0,   'color': '#95a5a6'},
    'H\n(中性策略)':    {'pass': 62.7, 'recall': 0.303, 'sacrifice': 86.3,  'color': '#e67e22'},
    'A\n(基准)':        {'pass': 62.5, 'recall': 0.346, 'sacrifice': 92.3,  'color': '#3498db'},
    'I\n(类别标签)':    {'pass': 70.5, 'recall': None,  'sacrifice': 92.7,  'color': '#9b59b6'},
    'J\n(删除日志)':    {'pass': 67.5, 'recall': None,  'sacrifice': 93.0,  'color': '#1abc9c'},
    'K\n(规格先写)':    {'pass': 81.9, 'recall': 0.268, 'sacrifice': 25.0,  'color': '#e74c3c'},
    'Oracle\n(GT assumes)': {'pass': 84.3, 'recall': 0.251, 'sacrifice': 11.8, 'color': '#c0392b'},
    'M\n(边界提示)':    {'pass': 75.3, 'recall': 0.389, 'sacrifice': 0.0,   'color': '#2ecc71'},
}

# 只用有 recall 数据的条件
conds_with_recall = {k: v for k, v in conditions_data.items() if v['recall'] is not None}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：PASS 率排序
ax = axes[0]
names = list(conds_with_recall.keys())
passes = [conds_with_recall[n]['pass'] for n in names]
colors = [conds_with_recall[n]['color'] for n in names]
sorted_by_pass = sorted(zip(passes, names, colors), reverse=True)
p_sorted, n_sorted, c_sorted = zip(*sorted_by_pass)

bars = ax.barh(range(len(n_sorted)), p_sorted, color=c_sorted, alpha=0.8, height=0.6)
ax.set_yticks(range(len(n_sorted)))
ax.set_yticklabels(n_sorted, fontsize=10)
ax.set_xlabel('PASS 率 (%)', fontsize=11)
ax.set_title('① PASS 率排序\n（越高越好？）', fontsize=12, fontweight='bold')
ax.set_xlim(0, 100)
for i, (p, n) in enumerate(zip(p_sorted, n_sorted)):
    ax.text(p + 1, i, f'{p:.1f}%', va='center', fontsize=9)
ax.axvline(50, color='gray', linestyle='--', alpha=0.4)

# 右图：Recall 排序（注意顺序反转！）
ax = axes[1]
recalls = [conds_with_recall[n]['recall'] for n in names]
sorted_by_recall = sorted(zip(recalls, names, colors), reverse=True)
r_sorted, n_sorted_r, c_sorted_r = zip(*sorted_by_recall)

bars = ax.barh(range(len(n_sorted_r)), [r*100 for r in r_sorted],
               color=c_sorted_r, alpha=0.8, height=0.6)
ax.set_yticks(range(len(n_sorted_r)))
ax.set_yticklabels(n_sorted_r, fontsize=10)
ax.set_xlabel('Recall (%) [相对 GT harness]', fontsize=11)
ax.set_title('② Recall 排序\n（实际规格覆盖率）', fontsize=12, fontweight='bold')
ax.set_xlim(0, 55)
for i, (r, n) in enumerate(zip(r_sorted, n_sorted_r)):
    ax.text(r*100 + 0.5, i, f'{r:.3f}', va='center', fontsize=9)

fig.suptitle('PASS 率 vs Recall：排名完全反转！\n'
             'Oracle (最高 PASS 84.3%) = 最低 Recall (0.251)；M (第三 75.3%) = 最高 Recall (0.389)',
             fontsize=12, fontweight='bold', color='#c0392b')
plt.tight_layout()
plt.show()

---
## Part 6：为什么排名会反转 — 三种机制

| 条件 | 高 PASS 的原因 | 低 Recall 的原因 |
|------|--------------|----------------|
| **Oracle** | GT assumes 提供了完美的状态约束 | LLM "懒" — assumes 太好了，任何简单断言都能通过，LLM 写了最少的断言 |
| **K** | NL 合约先写好，harness 从强约束出发 | NL 合约本身不完整，遗漏了 GT 的 postconditions |
| **A** | 迭代修复提高了 PASS | 部分正确断言在 UNKNOWN 压力下被删掉了（sacrifice）|
| **M** | 边界提示消除了 UNKNOWN | 没有删除行为，保留了所有正确断言 |

In [ ]:
# 散点图：PASS 率 vs Recall，显示反转关系
fig, ax = plt.subplots(figsize=(9, 6))

for name, d in conds_with_recall.items():
    short = name.replace('\n', ' ')
    ax.scatter(d['pass'], d['recall'] * 100, s=180, color=d['color'],
               zorder=5, alpha=0.9)
    offset_x = 1.5
    offset_y = 0.5
    if 'Oracle' in name:
        offset_x = -12; offset_y = 1
    elif 'M' in name:
        offset_x = 1; offset_y = 1
    ax.annotate(short,
                xy=(d['pass'], d['recall'] * 100),
                xytext=(d['pass'] + offset_x, d['recall'] * 100 + offset_y),
                fontsize=9.5, color=d['color'], fontweight='bold')

# 反趋势箭头
ax.annotate('', xy=(85, 24), xytext=(60, 36),
    arrowprops=dict(arrowstyle='->', color='#c0392b', lw=2, 
                    connectionstyle='arc3,rad=0.2'))
ax.text(70, 32, 'PASS ↑\nRecall ↓', color='#c0392b', fontsize=10,
        fontweight='bold', ha='center')

ax.set_xlabel('PASS 率 (%)', fontsize=12)
ax.set_ylabel('Recall (%) [相对 GT]', fontsize=12)
ax.set_title('PASS 率 vs Recall：验证成功 ≠ 规格完整\n'
             '理想情况下应该是正相关，但我们观察到负相关',
             fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_xlim(25, 92)
ax.set_ylim(22, 42)

# 理想线
ax.plot([30, 90], [26, 41], 'g--', alpha=0.3, label='理想：正相关')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## Part 7：Sacrifice 详解 — 两个 Gap 的区别

这是论文的核心叙事问题：
- **92.3% sacrifice ratio**：所有删除事件中，92.3% 是被 UNKNOWN 触发的（不是 FAIL）
- **97% never-generated**：GT 漏掉的 198 条断言中，97% 从来没有被 LLM 生成过

这不是矛盾！它们描述的是两个不同的 gap：

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左图：所有删除事件的触发原因（sacrifice ratio 的分母）
ax = axes[0]
labels = ['UNKNOWN 触发\n(Sacrifice)', 'FAIL 触发\n(Legitimate fix)', 'COMPILE_ERROR 触发']
sizes = [92.3, 4.8, 2.9]  # 近似，基于 A 条件
colors = ['#e74c3c', '#2ecc71', '#f39c12']
wedges, texts, autotexts = ax.pie(sizes, labels=labels, colors=colors,
                                   autopct='%1.1f%%', startangle=90,
                                   textprops={'fontsize': 10})
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
ax.set_title('Condition A 删除事件的触发原因\n（分母 = ALL deletions）',
             fontsize=11, fontweight='bold')

# 右图：GT 漏掉的 198 条断言的来源（97% 分析的分母）
ax = axes[1]

categories = [
    'Never generated\n(知识缺口, 97%)',
    'Deleted sacrifice\n(主动删除, 2.5%)',
    'Other/unclear\n(0.5%)'
]
values = [97.0, 2.5, 0.5]
colors_2 = ['#3498db', '#e74c3c', '#bdc3c7']
bars = ax.bar(categories, values, color=colors_2, alpha=0.85, width=0.6)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{v}%', ha='center', fontweight='bold', fontsize=12)
ax.set_ylabel('比例 (%)', fontsize=11)
ax.set_title('GT 漏掉的 198 条断言的来源分析\n（分母 = GT missed assertions）',
             fontsize=11, fontweight='bold')
ax.set_ylim(0, 108)
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle(
    '两个数字，两个分母，两个不同的问题\n'
    '92.3% sacrifice = "当 LLM 删断言时，几乎总是因为 UNKNOWN"\n'
    '97% never-generated = "GT 漏掉的断言，97% 是 LLM 从来不知道要写的"',
    fontsize=11, fontweight='bold', color='#2c3e50'
)
plt.tight_layout()
plt.show()

print()
print('📌 关键解释：')
print('  - 两个数字都是真实的，没有矛盾')
print('  - Sacrifice 主要发生在 LLM 自己过度生成的非 GT 断言上（删了也不坏）')
print('  - GT recall gap 主要来自 LLM 根本不知道要写什么（知识缺口）')
print('  - 但 sacrifice 导致的退化（H vs A 的 recall gap: Δ=5.6pp, p=0.026*）是可测量的')

---
## Part 8：11 个实验条件是什么，每个测什么

我们设计了 11 个条件来系统地**隔离**不同因素的影响。每个条件只改变一个变量。

In [ ]:
conditions_info = [
    ('G', '无反馈（单次生成）', '如果 LLM 只生成一次、不看 CBMC 结果，质量如何？\n→ CBMC 反馈循环的贡献', 31.3, 0.290, 0.0, '#95a5a6'),
    ('H', '中性策略\n（只给错误，不给策略）', 'CBMC UNKNOWN 时，LLM 是主动选择删断言，\n还是被我们的 prompt 引导的？\n→ Sacrifice 是涌现行为，不是被指令的', 62.7, 0.303, 86.3, '#e67e22'),
    ('A', '基准条件\n（标准反馈 prompt）', '标准的迭代 CBMC 反馈循环\n→ 所有其他条件与此对比', 62.5, 0.346, 92.3, '#3498db'),
    ('I', '类别标签\n（告诉 LLM 是什么类型断言）', '如果 LLM 知道要删的是 "frame condition"，\n它还会删吗？→ Sacrifice 是主动的，不是无知的', 70.5, None, 92.7, '#9b59b6'),
    ('J', '删除日志\n（显示历史删除记录）', '如果 LLM 记得"我之前删过什么"，\n它会停止删除吗？→ 不是遗忘造成的', 67.5, None, 93.0, '#1abc9c'),
    ('K', '规格先写\n（先写 NL 合约再生成断言）', '如果先写自然语言规格，\n会减少 sacrifice 吗？→ 是的，但 recall 更低（NL 合约本身不完整）', 81.9, 0.268, 25.0, '#e74c3c'),
    ('Oracle', 'GT Assumes 提供\n（用官方 assume）', '如果告诉 LLM 如何设置输入，\n它会写出更好的断言吗？→ 不会（懒惰效应）', 84.3, 0.251, 11.8, '#c0392b'),
    ('M', '边界提示\n（告诉 LLM 要 bound scalar）', '如果修复 LLM 对 CBMC 的知识缺口，\nUNKNOWN 会消失吗？→ 是的，sacrifice=0', 75.3, 0.389, 0.0, '#2ecc71'),
]

fig, ax = plt.subplots(figsize=(15, 8))
ax.axis('off')

col_headers = ['条件', '设计意图', '研究问题', 'PASS%', 'Recall', 'Sacrifice%']
col_x = [0.02, 0.10, 0.38, 0.68, 0.76, 0.86]
row_h = 0.095

# 表头
for x, h in zip(col_x, col_headers):
    ax.text(x, 0.97, h, fontsize=10, fontweight='bold', color='white',
            transform=ax.transAxes, va='top')
ax.add_patch(mpatches.FancyBboxPatch((0, 0.925), 1.0, 0.065,
    boxstyle='round,pad=0.005', facecolor='#2c3e50', transform=ax.transAxes))

for i, (cond, intent, question, pass_r, recall, sacr, color) in enumerate(conditions_info):
    y = 0.92 - (i + 1) * row_h
    bg_color = color + '15'
    ax.add_patch(mpatches.FancyBboxPatch((0, y - 0.005), 1.0, row_h,
        boxstyle='round,pad=0.005', facecolor=bg_color, transform=ax.transAxes,
        linewidth=0.5, edgecolor=color))
    
    yc = y + row_h * 0.5
    ax.text(col_x[0], yc, cond, fontsize=11, fontweight='bold', color=color,
            transform=ax.transAxes, va='center')
    ax.text(col_x[1], yc, intent, fontsize=8.5, color='#2c3e50',
            transform=ax.transAxes, va='center', linespacing=1.4)
    ax.text(col_x[2], yc, question, fontsize=8, color='#555',
            transform=ax.transAxes, va='center', linespacing=1.3,
            style='italic')
    ax.text(col_x[3], yc, f'{pass_r:.1f}%', fontsize=10, color=color,
            fontweight='bold', transform=ax.transAxes, va='center', ha='center')
    recall_str = f'{recall:.3f}' if recall else '—'
    ax.text(col_x[4], yc, recall_str, fontsize=10,
            color='#2ecc71' if recall and recall > 0.35 else '#e74c3c' if recall and recall < 0.3 else '#555',
            fontweight='bold', transform=ax.transAxes, va='center', ha='center')
    sacr_color = '#2ecc71' if sacr == 0.0 else ('#e74c3c' if sacr > 85 else '#f39c12')
    ax.text(col_x[5], yc, f'{sacr:.1f}%', fontsize=10, color=sacr_color,
            fontweight='bold', transform=ax.transAxes, va='center', ha='center')

ax.set_title('11 个实验条件总览：每个条件测什么问题', fontsize=13, fontweight='bold',
             pad=15)
plt.tight_layout()
plt.show()

---
## Part 9：Deletion_scope — Panic 删除 vs 精准删除

In [ ]:
# 来自 research_design.md 的 deletion_scope 数据
deletion_scope_data = {
    'A':      {'events': 29, 'mean_scope': 6.7, 'panic_pct': 72, 'targeted_pct': 17, 'color': '#3498db'},
    'I':      {'events': 24, 'mean_scope': 5.5, 'panic_pct': 71, 'targeted_pct': 12, 'color': '#9b59b6'},
    'J':      {'events': 21, 'mean_scope': 5.6, 'panic_pct': 76, 'targeted_pct': 10, 'color': '#1abc9c'},
    'K':      {'events': 5,  'mean_scope': 1.6, 'panic_pct': 20, 'targeted_pct': 80, 'color': '#e74c3c'},
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 左图：UNKNOWN 事件次数
ax = axes[0]
cnames = list(deletion_scope_data.keys())
events = [deletion_scope_data[c]['events'] for c in cnames]
colors_ds = [deletion_scope_data[c]['color'] for c in cnames]
ax.bar(cnames, events, color=colors_ds, alpha=0.8, width=0.5)
ax.set_ylabel('UNKNOWN 触发的删除事件数')
ax.set_title('UNKNOWN 删除事件次数')
for i, (c, e) in enumerate(zip(cnames, events)):
    ax.text(i, e + 0.3, str(e), ha='center', fontweight='bold')
ax.set_ylim(0, 35)

# 中图：平均每次删多少断言
ax = axes[1]
means = [deletion_scope_data[c]['mean_scope'] for c in cnames]
ax.bar(cnames, means, color=colors_ds, alpha=0.8, width=0.5)
ax.set_ylabel('平均删除断言数/事件')
ax.set_title('每次 UNKNOWN → 平均删多少断言\n（A 最差：平均删 6.7 条，最多一次删了 20 条！)')
for i, (c, m) in enumerate(zip(cnames, means)):
    ax.text(i, m + 0.1, f'{m:.1f}', ha='center', fontweight='bold')
ax.set_ylim(0, 9)
ax.axhline(1, color='green', linestyle='--', alpha=0.5, label='精准删除（scope=1）')
ax.legend(fontsize=8)

# 右图：Panic vs Targeted 比例
ax = axes[2]
panic = [deletion_scope_data[c]['panic_pct'] for c in cnames]
targeted = [deletion_scope_data[c]['targeted_pct'] for c in cnames]
x = np.arange(len(cnames))
w = 0.35
bars1 = ax.bar(x - w/2, panic, width=w, label='Panic (scope≥3)',
               color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + w/2, targeted, width=w, label='Targeted (scope=1)',
               color='#2ecc71', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(cnames)
ax.set_ylabel('%')
ax.set_title('Panic 删除 vs 精准删除\n（K 条件因为有 NL 合约作为参考，80% 精准）')
ax.legend()
ax.set_ylim(0, 95)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.0f}%', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.0f}%', ha='center', fontsize=9)

fig.suptitle('Deletion Scope 分析：UNKNOWN 触发时，LLM 怎么删断言',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n解读：')
print('  A/I/J：72-76% 是 panic 删除——CBMC 超时了，LLM 不知道哪个断言有问题，索性大扫清')
print('  K：   80% 精准删除——因为有自己写的 NL 合约作参考，能定位到具体哪条断言冲突')
print('  M：   0 次删除——CBMC 不再 UNKNOWN，没有删除压力')

---
## Part 10：统计显著性

用 Wilcoxon signed-rank 检验验证 recall 差异是否在统计上显著（在 83 个函数上的配对检验）。

In [ ]:
from scipy.stats import wilcoxon

def load_recall(path):
    with open(path) as f:
        entries = json.load(f)
    return {e['func']: (e['harness_recall'] if e['harness_recall'] is not None else 0.0)
            for e in entries if e['gt_harness_count'] > 0}

cond_paths = {
    'A':      os.path.join(EVAL, 'cross_verify_results_condA_gptoss120b.json'),
    'H':      os.path.join(EVAL, 'cross_verify_results_condH_gptoss120b.json'),
    'G':      os.path.join(EVAL, 'cross_verify_results_condG_gptoss120b.json'),
    'K':      os.path.join(EVAL, 'cross_verify_results_condK_gptoss120b.json'),
    'Oracle': os.path.join(EVAL, 'cross_verify_results_condOracle_gptoss120b.json'),
    'M':      os.path.join(EVAL, 'cross_verify_results_condM_gptoss120b.json'),
}

recalls = {k: load_recall(v) for k, v in cond_paths.items()}

# 关键比较
tests = [
    ('M', 'Oracle', '懒惰效应反转：M > Oracle'),
    ('M', 'K',      'PASS-Recall 反转：M > K'),
    ('M', 'H',      '边界提示效果：M > H'),
    ('A', 'Oracle', '懒惰效应：A > Oracle（有 GT assumes 反而更差）'),
    ('A', 'K',      '规格先写不提升 recall：A > K'),
    ('A', 'H',      '策略指导有效：A > H（同等 PASS 下）'),
    ('M', 'A',      'M vs A（微弱，ns）'),
]

print(f'{"比较":<30} {"n":>4} {"Δmean recall":>13} {"p (one-sided)":>15} {"显著性":>8}')
print('-'*75)

results_for_plot = []
for c1, c2, label in tests:
    shared = sorted(set(recalls[c1]) & set(recalls[c2]))
    v1 = np.array([recalls[c1][f] for f in shared])
    v2 = np.array([recalls[c2][f] for f in shared])
    diff = v1 - v2
    nonzero = np.sum(diff != 0)
    mean_diff = np.mean(v1) - np.mean(v2)
    if nonzero >= 6:
        stat, p = wilcoxon(v1, v2, alternative='greater')
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        results_for_plot.append((label[:35], mean_diff, p, sig))
        print(f'{label:<45} {len(shared):>4} {mean_diff:>+13.4f} {p:>15.4f} {sig:>8}')

print()
print('* p<0.05, ** p<0.01, *** p<0.001 (Wilcoxon signed-rank, one-sided, per-function strict recall)')

In [ ]:
# 可视化统计显著性
fig, ax = plt.subplots(figsize=(12, 5))

labels_plot = [r[0] for r in results_for_plot]
diffs = [r[1] * 100 for r in results_for_plot]  # 转为百分点
pvals = [r[2] for r in results_for_plot]
sigs = [r[3] for r in results_for_plot]

bar_colors = ['#2ecc71' if d > 0 else '#e74c3c' for d in diffs]
alpha_vals = [1.0 if s != 'ns' else 0.4 for s in sigs]

bars = ax.barh(range(len(labels_plot)), diffs, color=bar_colors, alpha=0.8, height=0.6)
ax.set_yticks(range(len(labels_plot)))
ax.set_yticklabels(labels_plot, fontsize=9.5)
ax.set_xlabel('Δ Recall (百分点)', fontsize=11)
ax.set_title('各条件 recall 差异的 Wilcoxon 检验结果', fontsize=13, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8)

for i, (d, p, s) in enumerate(zip(diffs, pvals, sigs)):
    x_pos = d + (0.1 if d >= 0 else -0.1)
    ha = 'left' if d >= 0 else 'right'
    color = '#2ecc71' if s in ['*', '**', '***'] else '#888'
    ax.text(x_pos, i, f'p={p:.4f} {s}', va='center', ha=ha,
            fontsize=9, fontweight='bold', color=color)

ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

---
## Part 11：Vacuity Check — 我们的结果是真实的吗？

**担心**：K 和 Oracle 的高 PASS 率会不会是「虚假成功」（vacuous success）——即 `__CPROVER_assume` 过于严格，把所有输入都过滤掉了，导致 CBMC 永远不会到达函数调用，从而断言永远不被评估（trivially UNSAT）？

**验证方法**：在函数调用前插入 `__CPROVER_assert(false)`。如果 CBMC 仍然返回 UNSAT，说明函数调用路径不可达 = 真的 vacuous。

In [ ]:
# 读取 vacuity 检验结果
vacuity_summary = {}
for cond in ['K', 'Oracle', 'M']:
    path = os.path.join(EVAL, f'vacuity_check_{cond}_gptoss120b.json')
    with open(path) as f:
        entries = json.load(f)
    total = len(entries)
    vacuous = [e['func'] for e in entries if e['vacuous'] is True]
    reachable = [e['func'] for e in entries if e['vacuous'] is False]
    unknown = [e['func'] for e in entries if e['vacuous'] is None]
    vacuity_summary[cond] = {
        'total': total,
        'vacuous': vacuous,
        'reachable': len(reachable),
        'unknown': len(unknown),
    }

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, (cond, data) in zip(axes, vacuity_summary.items()):
    total = data['total']
    sizes = [len(data['vacuous']), data['reachable'], data['unknown']]
    labels_v = [
        f'Vacuous\n({len(data["vacuous"])}/{total}, {100*len(data["vacuous"])/total:.1f}%)',
        f'Reachable (genuine)\n({data["reachable"]}/{total})',
        f'Unknown/Timeout\n({data["unknown"]}/{total})'
    ]
    colors_v = ['#e74c3c', '#2ecc71', '#bdc3c7']
    wedges, texts = ax.pie(sizes, labels=labels_v, colors=colors_v,
                           startangle=90, textprops={'fontsize': 8.5})
    title_color = '#2ecc71' if len(data['vacuous']) == 0 else '#e74c3c'
    ax.set_title(f'Condition {cond}\n(n={total} SUCCESS harnesses)',
                 fontsize=11, fontweight='bold', color=title_color)
    if data['vacuous']:
        ax.text(0, -1.35, f'Vacuous function: {data["vacuous"][0]}',
                ha='center', fontsize=8, color='#e74c3c', style='italic')

fig.suptitle(
    'Vacuity Check 结果\n'
    'K=1.5% vacuous, Oracle=1.4% vacuous, M=0% vacuous\n'
    '→ PASS-Recall 反转不是 vacuous success 的伪影，是真实的规格质量差距',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

---
## Part 12：整体发现总结

研究的故事线：

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.axis('off')

story = [
    ('①', '基本现象',
     'A 条件：92.3% 的断言删除是被 CBMC UNKNOWN 触发的，\n'
     '不是因为断言错了（那应该是 FAIL 触发）。\n'
     '→ LLM 在帮 CBMC 验证，不在写正确的规格。',
     '#3498db'),
    ('②', '涌现性验证\n(H 条件)',
     'H 条件不告诉 LLM "你可以删断言"，\n'
     '但 sacrifice 率依然达到 86.3%，recall 比 A 低 5.6pp (p=0.026*)。\n'
     '→ Sacrifice 是 LLM 在 CBMC 压力下的主动策略，不是被指令的。',
     '#e67e22'),
    ('③', '故意性验证\n(I 条件)',
     '告诉 LLM "你要删的是 frame condition"，sacrifice 反而升到 92.7%，\n'
     '且删除变得更精准（panic→targeted）。\n'
     '→ LLM 知道自己在删什么，仍然选择删——这是有意识的 conformance 行为。',
     '#9b59b6'),
    ('④', 'PASS-Recall 反转\n（核心发现）',
     'PASS 率最高的 Oracle（84.3%）recall 最低（0.251）。\n'
     'PASS 率第三的 M（75.3%）recall 最高（0.389）。\n'
     '→ 验证通过率是误导性指标；知识修复比提供更好的脚手架更有效。',
     '#c0392b'),
    ('⑤', '知识缺口 vs 退化缺口',
     '97% 的 GT miss 来自「从未生成过」（知识缺口）。\n'
     '但 H vs A 的 recall 差（5.6pp, p=0.026*）是可测量的「退化缺口」。\n'
     '→ 两个独立的问题，需要不同的解决方案。',
     '#2c3e50'),
    ('⑥', '唯一正面结果\n(M 条件)',
     '告诉 LLM "在 CBMC 中要 bound scalar variables"，\n'
     'UNKNOWN 事件归零，sacrifice=0，recall 提升到 0.389（最高）。\n'
     '→ 一个简单的 CBMC 工具知识修复，同时提升了验证率和规格质量。',
     '#2ecc71'),
]

for i, (num, title, detail, color) in enumerate(story):
    y = 0.88 - i * 0.145
    ax.add_patch(mpatches.FancyBboxPatch((0.01, y - 0.065), 0.97, 0.12,
        boxstyle='round,pad=0.01', linewidth=1.5,
        edgecolor=color, facecolor=color + '15',
        transform=ax.transAxes))
    ax.text(0.03, y, num, fontsize=16, fontweight='bold', color=color,
            transform=ax.transAxes, va='center')
    ax.text(0.08, y + 0.025, title, fontsize=10.5, fontweight='bold', color=color,
            transform=ax.transAxes, va='center')
    ax.text(0.08, y - 0.025, detail, fontsize=8.8, color='#2c3e50',
            transform=ax.transAxes, va='center', linespacing=1.5)

ax.set_title('研究发现的故事线：从现象到机制到干预',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---
## Part 13：接下来要做什么 (RQ2)

目前 RQ1 回答了「LLM 漏掉了什么，为什么」。但 **RQ2** 回答的是：**那些漏掉的断言，真的会让 bug 漏网吗？**

方法：**Mutation Oracle**
- 用 universalmutator 对 aws-c-common 的 83 个函数注入 2584 个 bug（mutant）
- 分别跑 GT harness 和 LLM harness
- 统计「GT 能发现但 LLM 发现不了」的 mutant 数量 → **silenced mutant count**

这个数字是整篇论文的核心结果——它把 recall gap 从「规格对比」变成「实际 bug 被漏掉」。

In [ ]:
# 检查 mutant 数量
mutant_dir = os.path.join(RESULTS, 'feedback_loop_A_gptoss120b')
# 找每个函数的 mutant 文件
total_mutants = 0
func_mutant_counts = {}
for func in os.listdir(mutant_dir):
    func_path = os.path.join(mutant_dir, func)
    if not os.path.isdir(func_path):
        continue
    mutants = [f for f in os.listdir(func_path) if f.endswith('.mutant.c')]
    if mutants:
        func_mutant_counts[func] = len(mutants)
        total_mutants += len(mutants)

print(f'已生成 mutant 数量：{total_mutants:,} 个，分布在 {len(func_mutant_counts)} 个函数')
print()
print('Mutant 数量最多的函数：')
for func, n in sorted(func_mutant_counts.items(), key=lambda x: -x[1])[:8]:
    bar = '█' * (n // 5)
    print(f'  {func:<40} {n:>4}  {bar}')

print()
print('下一步：')
print('  1. ESBMC parity check：在 GT harness 上验证 ESBMC 能正确运行')
print('  2. 对 2584 个 mutant 跑 ESBMC，找出 GT 能抓到但 LLM 抓不到的')
print('  3. 这个数字 = silenced mutant count = 论文的核心安全性主张')

---
## 总结

| 实验步骤 | 在测什么 | 关键发现 |
|---------|---------|--------|
| CBMC 反馈循环 | LLM 能否在迭代中生成正确 harness | 能通过验证，但会删掉正确断言 |
| Sacrifice 分析 | 删除断言的触发原因 | 92.3% 是 UNKNOWN 触发，不是错误 |
| 条件消融 (I/J) | Sacrifice 是主动还是被动 | 主动的——知道也删，有了记录还删 |
| PASS vs Recall | PASS 率能代表规格质量吗 | 不能——两者负相关 |
| Vacuity check | K/Oracle 的高 PASS 是虚假的吗 | 不是——只有 1-2% vacuous |
| M 条件 | 修复知识缺口能解决问题吗 | 能——M 是唯一同时提升两项指标的条件 |
| **RQ2（待做）** | **漏掉的断言真的让 bug 逃逸了吗** | **待测——2584 个 mutant 准备好了** |

**核心论点**：LLMs are optimizing for verifier satisfaction, not specification completeness.
验证通过 ≠ 规格正确；这个偏差是系统性的、有意识的、可量化的。